# Neural generators (v0.5)

ML-assisted chord and melody generation using small LSTMs trained on a self-generated corpus.

**Workflow:**

```
musicgen corpus  →  extract-sequences  →  train  →  generate --chord-backend neural
```

**Requires:** `pip install -e '.[neural]'` (installs `torch >= 2.0`). FluidSynth needed only for end-to-end generation (Sections 6 + 7).

**Cell tags:**
- `# requires: torch` — needs `musicgen[neural]` extra
- `# requires: fluidsynth` — needs FluidSynth binary

## 0. Cloud setup (Colab / Binder)

Run this cell first when launching on Google Colab or mybinder.org. On a local install it's a no-op.

In [ ]:
# Colab/Binder bootstrap — skipped on local installs
import sys, os, shutil

IN_COLAB = "google.colab" in sys.modules
IN_BINDER = "JUPYTERHUB_USER" in os.environ or os.path.exists("/home/jovyan")

if IN_COLAB:
    # apt deps (fluidsynth binary + FluidR3_GM soundfont)
    !apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg
    # musicgen + neural extra
    !pip install -q "musicgen[neural] @ git+https://github.com/dobidu/layered_music_gen.git"
    # Clone repo for config.py + genres/ + notebooks (sys.path needs repo root)
    if not os.path.exists("/content/musicgen_repo"):
        !git clone -q https://github.com/dobidu/layered_music_gen.git /content/musicgen_repo
    os.chdir("/content/musicgen_repo")
    # Link the soundfont into per-layer sf dirs
    SF2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    for layer in ("beat", "melody", "harmony", "bassline"):
        os.makedirs(f"sf/{layer}", exist_ok=True)
        dst = f"sf/{layer}/FluidR3_GM.sf2"
        if not os.path.exists(dst):
            os.symlink(SF2, dst)
    print("Colab setup complete. Repo root:", os.getcwd())
elif IN_BINDER:
    print("Binder env detected — apt deps + soundfonts handled by binder/postBuild")
else:
    print("Local env — assumed to have FluidSynth + .sf2 files installed already")


## 1. Setup

In [ ]:
import shutil, sys, os, json, tempfile, pathlib, random

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config import Config
from musicgen import __version__

HAS_FLUIDSYNTH = shutil.which("fluidsynth") is not None
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

print(f"musicgen {__version__}")
print(f"FluidSynth present: {HAS_FLUIDSYNTH}")
print(f"torch present:      {HAS_TORCH}")
if HAS_TORCH:
    print(f"torch version:      {torch.__version__}")

## 2. Synthesize a tiny `sequences.json` corpus

For demo purposes we hand-craft a few chord and melody sequences. In real use you'd run `musicgen extract-sequences --dataset <corpus> --output sequences.json` on a directory of musicgen-generated samples.

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed — run pip install 'musicgen[neural]'")
else:
    GENRES = ["pop", "jazz"]
    sequences = {
        "metadata": {"n_samples": 20, "genres": GENRES,
                     "musicgen_version": __version__, "n_skipped": 0},
        "chord": [
            {"sample_index": i, "genre": [GENRES[i % 2]], "key": "C",
             "full_sequence": ["I", "V", "vi", "IV", "I", "IV", "V", "I"] * 4}
            for i in range(40)
        ],
        "melody": [
            {"sample_index": i, "genre": [GENRES[i % 2]], "key": "C",
             "full_sequence": ["1", "2", "3", "4", "5", "4", "3", "2"] * 4}
            for i in range(40)
        ],
    }

    work_dir = tempfile.mkdtemp(prefix="musicgen_neural_")
    seq_path = os.path.join(work_dir, "sequences.json")
    pathlib.Path(seq_path).write_text(json.dumps(sequences))
    print(f"Wrote demo corpus → {seq_path}")
    print(f"  chord sequences:  {len(sequences['chord'])}")
    print(f"  melody sequences: {len(sequences['melody'])}")
    print(f"  genres:           {GENRES}")

## 3. Inspect the model architecture

`ChordLSTM` (~35K params) and `MelodyLSTM` (~10K params) share the `_SequenceLSTM` base. Each takes a context window of length `context_len` and a genre id, returns logits over vocab.

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed")
else:
    from musicgen.neural.model import ChordLSTM, MelodyLSTM

    chord_model = ChordLSTM(vocab_size=20, genre_count=5)
    melody_model = MelodyLSTM(vocab_size=8, genre_count=5)

    def n_params(m):
        return sum(p.numel() for p in m.parameters())

    print(f"ChordLSTM(vocab=20, genres=5) params: {n_params(chord_model):,}")
    print(f"MelodyLSTM(vocab=8, genres=5) params: {n_params(melody_model):,}")
    print()
    print("ChordLSTM modules:")
    print(chord_model)

## 4. Train chord + melody models

`train()` runs Adam + CrossEntropyLoss(ignore_index=`<pad>`) with patience-based early stopping on a 90/10 train/val split. CPU is fine for this corpus size.

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed")
else:
    from musicgen.neural.trainer import train, save_model

    chord_sampler = train(seq_path, layer="chord", epochs=30, seed=0)
    melody_sampler = train(seq_path, layer="melody", epochs=30, seed=0)

    models_dir = os.path.join(work_dir, "models")
    os.makedirs(models_dir, exist_ok=True)
    save_model(chord_sampler,  os.path.join(models_dir, "chord.pt"))
    save_model(melody_sampler, os.path.join(models_dir, "melody.pt"))

    print(f"\nSaved models to {models_dir}")
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f))
        print(f"  {f}  ({size} bytes)")

## 5. Sample from the trained models

Same seeded `random.Random` → identical output. The determinism contract is preserved end-to-end.

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed")
else:
    from musicgen.neural.sampler import sample_chord_neural, sample_melody_neural

    # Generate a chord progression
    history = []
    rng = random.Random(42)
    for _ in range(8):
        chord = sample_chord_neural(history, ["pop"], chord_sampler, rng)
        history.append(chord)
    print(f"Chord progression (pop, seed=42): {history}")

    # Re-run with same seed → same output
    h2 = []
    rng2 = random.Random(42)
    for _ in range(8):
        h2.append(sample_chord_neural(h2, ["pop"], chord_sampler, rng2))
    print(f"Re-run with same seed:           {h2}")
    print(f"Deterministic: {history == h2}")

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed")
else:
    # Melody — sample 16 scale degrees
    melody_history = []
    rng = random.Random(7)
    for _ in range(16):
        deg = sample_melody_neural(melody_history, ["jazz"], melody_sampler, rng)
        melody_history.append(deg)
    print(f"Melody scale-degree sequence (jazz, seed=7): {melody_history}")

## 6. End-to-end generation with neural backends

Plug the trained models into the full pipeline via `Config.chord_backend = 'neural'`.

In [ ]:
# requires: fluidsynth + torch
if not (HAS_FLUIDSYNTH and HAS_TORCH):
    print("Skipping: requires FluidSynth + torch")
else:
    from musicgen import generate

    with tempfile.TemporaryDirectory(prefix="musicgen_neural_gen_") as out:
        cfg = Config.load(cli_overrides={
            "global_seed": 42,
            "sample_index": 0,
            "dataset_root": out,
            "chord_backend": "neural",
            "melody_backend": "neural",
            "models_dir": models_dir,
        })
        result = generate(cfg)
        print(f"status: {result.status}")
        print(f"musicality: {result.musicality_score:.3f}")
        print(f"sample_dir: {result.sample_dir}")

        sj = json.loads(pathlib.Path(result.sample_json_path).read_text())
        print(f"\nFirst-part chord progression: {sj.get('chord_progression', {}).get(list(sj.get('chord_progression', {}))[0], [])[:8]}")

## 7. End-to-end determinism check

Same seed + same models → bit-identical `sample.json` across runs.

In [ ]:
# requires: fluidsynth + torch
if not (HAS_FLUIDSYNTH and HAS_TORCH):
    print("Skipping: requires FluidSynth + torch")
else:
    import hashlib
    from musicgen import generate

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                h.update(chunk)
        return h.hexdigest()

    hashes = []
    for run in range(2):
        with tempfile.TemporaryDirectory() as out:
            cfg = Config.load(cli_overrides={
                "global_seed": 99,
                "sample_index": 0,
                "dataset_root": out,
                "chord_backend": "neural",
                "melody_backend": "neural",
                "models_dir": models_dir,
            })
            r = generate(cfg)
            hashes.append(sha256_file(r.sample_json_path))

    print(f"Run 1 sha256: {hashes[0][:16]}...")
    print(f"Run 2 sha256: {hashes[1][:16]}...")
    print(f"Determinism: {'PASS' if hashes[0] == hashes[1] else 'FAIL'}")

## 8. Fallback when model is missing

Pointing `models_dir` at a non-existent directory while `chord_backend='neural'` is set logs a warning and falls back to Markov silently — generation still succeeds.

In [ ]:
# requires: fluidsynth
if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    from musicgen import generate
    import logging
    logging.basicConfig(level=logging.WARNING)

    with tempfile.TemporaryDirectory() as out:
        cfg = Config.load(cli_overrides={
            "global_seed": 1,
            "sample_index": 0,
            "dataset_root": out,
            "chord_backend": "neural",      # neural requested
            "models_dir": "/nonexistent",   # model file absent → fallback
        })
        r = generate(cfg)
        print(f"\nstatus: {r.status}  (fell back to Markov)")

## 9. Genre-specific models

Train one model per genre and the inference loop will prefer `chord_{genre}.pt` over the combined `chord.pt`.

In [ ]:
# requires: torch
if not HAS_TORCH:
    print("Skipping: torch not installed")
else:
    from musicgen.neural.trainer import train, save_model

    pop_sampler = train(seq_path, layer="chord", genres=["pop"], epochs=20, seed=1)
    jazz_sampler = train(seq_path, layer="chord", genres=["jazz"], epochs=20, seed=1)

    save_model(pop_sampler,  os.path.join(models_dir, "chord_pop.pt"))
    save_model(jazz_sampler, os.path.join(models_dir, "chord_jazz.pt"))

    print("Genre-specific models in models_dir:")
    for f in sorted(os.listdir(models_dir)):
        if f.endswith(".pt"):
            print(f"  {f}")
    print("\nAt inference, cfg.genre=['pop'] → chord_pop.pt wins over chord.pt")

## 10. Cleanup

In [ ]:
import shutil
try:
    shutil.rmtree(work_dir, ignore_errors=True)
    print(f"Removed {work_dir}")
except NameError:
    pass

## Further reading

- [`docs/neural-generators.md`](../docs/neural-generators.md) — full reference (CLI, schema, architecture, hyperparameters)
- README section: **ML-assisted generators (v0.5)**
- `tests/test_neural_model.py`, `tests/test_neural_integration.py` — minimal usage examples